# Stage 5 — Final Earth network generation and QA

_Pipeline stage 5 of `docs/PIPELINE_DESIGN.md`. Uses the canonical `channel_heads` package + on-disk artifacts; heavy rebuilds run via the `channel-heads` CLI (see each stage's command)._

## Building & QA-gating the Earth feature tables

Per regime we build every basin's regime-pruned network, enumerate confluence pairs
per outlet, and compute features + the touching label — a **QA gate** that must pass
(0 hard flags) before training. Build:
`python -m channel_heads build-earth-features --regime <r>`. We load the QA report
and confirm it is clean.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import channel_heads as ch
from channel_heads.io.paths import PROJECT_ROOT, RESULTS_DIR, EXAMPLE_DEMS
ROOT     = PROJECT_ROOT
MODELS   = ROOT / 'models'
MARS_OUT = ROOT / 'data/Mars/model_outputs'
MARS_IN  = ROOT / 'data/Mars/model_inputs'
REGIMES_ = ['regA', 'regB', 'regC']
# Frozen model features (5 dimensionless) + operating thresholds (Stage 9).
MODEL_FEATURES = ['orientation_diff_deg','headhead_dist_norm','apex_angle_deg',
                  'strahler_order_diff','proximity_profile_norm']
OP_THR = {'regA': 0.756326, 'regB': 0.779264, 'regC': 0.759369}
print('channel_heads', ch.__version__, '| root', ROOT)


channel_heads 0.1.0 | root /Users/guypi/Projects/channel-heads


In [2]:
qa = RESULTS_DIR / 'stage5_earth_network_qa_report.csv'
if qa.exists():
    q = pd.read_csv(qa)
    print('QA rows:', len(q), '| columns:', list(q.columns))
    for c in [c for c in q.columns if 'flag' in c.lower()]:
        n = q[c].sum() if q[c].dtype != object else (q[c].astype(str).str.len() > 0).sum()
        print(f'  {c}: {int(n)} flagged')
    display(q.head(20))
else:
    print('QA report not found - run build-earth-features first.')

QA rows: 17 | columns: ['basin', 'regA_pairs', 'regA_touching', 'regA_error', 'regA_ratio', 'regB_pairs', 'regB_touching', 'regB_error', 'regB_ratio', 'regC_pairs', 'regC_touching', 'regC_error', 'regC_ratio', 'paper_name', 'has_warning']


,basin,regA_pairs,regA_touching,regA_error,regA_ratio,regB_pairs,regB_touching,regB_error,regB_ratio,regC_pairs,regC_touching,regC_error,regC_ratio,paper_name,has_warning
0,calnalpine,23,10,NaN,0.434783,15,6,NaN,0.400000,78,7,NaN,0.089744,clanalpine,False
1,daqing,25,17,NaN,0.680000,19,13,NaN,0.684211,92,8,NaN,0.086957,daqing,False
2,finisterre,7892,1544,NaN,0.195641,4169,862,NaN,0.206764,15776,475,NaN,0.030109,finisterre,True
3,humboldt,33,12,NaN,0.363636,21,7,NaN,0.333333,144,13,NaN,0.090278,humboldt,False
4,inyo,60,24,NaN,0.400000,33,9,NaN,0.272727,178,26,NaN,0.146067,inyo,False
5,kammanasie,69,29,NaN,0.420290,31,13,NaN,0.419355,213,16,NaN,0.075117,kammanassie,False
6,luliang,153,60,NaN,0.392157,98,29,NaN,0.295918,544,25,NaN,0.045956,luliang,True
7,panamint,284,68,NaN,0.239437,152,47,NaN,0.309211,732,46,NaN,0.062842,panamint,False
8,sakhalin,176,55,NaN,0.312500,83,25,NaN,0.301205,444,30,NaN,0.067568,sakhalin,False
9,sierramadre,8945,1838,NaN,0.205478,4227,849,NaN,0.200852,18091,532,NaN,0.029407,sierramadre,True


Deep dive: `notebooks/analysis/05_earth_network_qa.ipynb`.